# March Madness 2026 — XGBoost Prediction Model

In [1]:
import pandas as pd
import numpy as np
import requests
import io
import json
import warnings
warnings.filterwarnings('ignore')

# ── Feature lists ──────────────────────────────────────────────
BASE_FEATURES      = ['adjoe', 'adjde', 'barthag', 'sos', 'WAB', 'ncsos', 'consos', 'Opp OE', 'Opp DE']
FFFINAL_FEATURES   = ['eFG%', 'FTR', 'OR%', 'DR%', 'TO%']
VARIANCE_FEATURES  = ['oe_std', 'de_std']
PRETOURNEY_FEATURES= ['pre_adjoe', 'pre_adjde', 'pre_barthag', 'pre_sos', 'pre_WAB']
TORVIK_FEATURES    = BASE_FEATURES  # alias for merges

ALL_FEATURES = BASE_FEATURES + FFFINAL_FEATURES + VARIANCE_FEATURES + PRETOURNEY_FEATURES
FEATURE_COLS_RESID = ([f'{f}_diff' for f in BASE_FEATURES + FFFINAL_FEATURES + PRETOURNEY_FEATURES]
                      + ['oe_std_resid_diff', 'de_std_resid_diff', 'seed_diff'])

## 1. Load Torvik season data

In [2]:
torvik_dfs = []
for year in range(2010, 2027):
    path = f"/Users/arnavgupta/Desktop/mm26/data/torvik/{year}_team_results.csv"
    df = pd.read_csv(path, quotechar='"', on_bad_lines='skip', index_col=False)
    df['year'] = year
    torvik_dfs.append(df)
    print(f"Loaded {year} — {len(df)} rows, first team: {df['team'].iloc[0]}")

torvik = pd.concat(torvik_dfs, ignore_index=True)
torvik_clean = torvik[['team', 'year'] + TORVIK_FEATURES].copy()
print(f"\nDone! torvik shape: {torvik.shape}")
print(f"torvik_clean columns: {torvik_clean.columns.tolist()}")

Loaded 2010 — 347 rows, first team: Duke
Loaded 2011 — 345 rows, first team: Ohio St.
Loaded 2012 — 345 rows, first team: Kentucky
Loaded 2013 — 347 rows, first team: Louisville
Loaded 2014 — 351 rows, first team: Louisville
Loaded 2015 — 351 rows, first team: Kentucky
Loaded 2016 — 351 rows, first team: Villanova
Loaded 2017 — 351 rows, first team: Gonzaga
Loaded 2018 — 351 rows, first team: Villanova
Loaded 2019 — 353 rows, first team: Gonzaga
Loaded 2020 — 353 rows, first team: Kansas
Loaded 2021 — 347 rows, first team: Gonzaga
Loaded 2022 — 358 rows, first team: Gonzaga
Loaded 2023 — 363 rows, first team: Connecticut
Loaded 2024 — 362 rows, first team: Connecticut
Loaded 2025 — 364 rows, first team: Houston
Loaded 2026 — 365 rows, first team: Duke

Done! torvik shape: (6004, 47)
torvik_clean columns: ['team', 'year', 'adjoe', 'adjde', 'barthag', 'sos', 'WAB', 'ncsos', 'consos', 'Opp OE', 'Opp DE']


## 2. Load Kaggle tournament data

In [3]:
results = pd.read_csv("/Users/arnavgupta/Desktop/mm26/data/kaggle/MNCAATourneyCompactResults.csv")
seeds   = pd.read_csv("/Users/arnavgupta/Desktop/mm26/data/kaggle/MNCAATourneySeeds.csv")
teams   = pd.read_csv("/Users/arnavgupta/Desktop/mm26/data/kaggle/MTeams.csv")

seeds['SeedNum'] = seeds['Seed'].str.extract(r'(\d+)').astype(int)
print("Tournament results:", results.shape)
print("Seeds:", seeds.shape)
print("Teams:", teams.shape)

Tournament results: (2585, 8)
Seeds: (2694, 4)
Teams: (381, 4)


## 3. Name mapping (Kaggle → Torvik)

In [4]:
name_map = {
    'Abilene Chr': 'Abilene Christian', 'Alabama St': 'Alabama St.',
    'Alcorn St': 'Alcorn St.', 'American Univ': 'American',
    'Appalachian St': 'Appalachian St.', 'Arizona St': 'Arizona St.',
    'Ark Little Rock': 'Little Rock', 'Ark Pine Bluff': 'Arkansas Pine Bluff',
    'Arkansas St': 'Arkansas St.', 'Ball St': 'Ball St.',
    'Bethune-Cookman': 'Bethune Cookman', 'Birmingham So': 'Birmingham Southern',
    'Boise St': 'Boise St.', 'Boston Univ': 'Boston University',
    'Brooklyn': 'LIU Brooklyn', 'C Michigan': 'Central Michigan',
    'CS Bakersfield': 'Cal St. Bakersfield', 'CS Fullerton': 'Cal St. Fullerton',
    'CS Northridge': 'Cal St. Northridge', 'CS Sacramento': 'Sacramento St.',
    'Cent Arkansas': 'Central Arkansas', 'Central Conn': 'Central Connecticut',
    'Charleston So': 'Charleston Southern', 'Chicago St': 'Chicago St.',
    'Citadel': 'The Citadel', 'Cleveland St': 'Cleveland St.',
    'Coastal Car': 'Coastal Carolina', 'Col Charleston': 'Col. of Charleston',
    'Colorado St': 'Colorado St.', 'Coppin St': 'Coppin St.',
    'Delaware St': 'Delaware St.', 'Detroit': 'Detroit Mercy',
    'E Illinois': 'Eastern Illinois', 'E Kentucky': 'Eastern Kentucky',
    'E Michigan': 'Eastern Michigan', 'E Washington': 'Eastern Washington',
    'ETSU': 'East Tennessee St.', 'F Dickinson': 'Fairleigh Dickinson',
    'FGCU': 'Florida Gulf Coast', 'FL Atlantic': 'Florida Atlantic',
    'Florida Intl': 'Florida International', 'Florida St': 'Florida St.',
    'Fresno St': 'Fresno St.', 'G Washington': 'George Washington',
    'Ga Southern': 'Georgia Southern', 'Georgia St': 'Georgia St.',
    'Grambling': 'Grambling St.', 'Idaho St': 'Idaho St.',
    'Illinois St': 'Illinois St.', 'Indiana St': 'Indiana St.',
    'Iowa St': 'Iowa St.', 'Jackson St': 'Jackson St.',
    'Jacksonville St': 'Jacksonville St.', 'Kansas St': 'Kansas St.',
    'Kennesaw': 'Kennesaw St.', 'Kent': 'Kent St.',
    'Long Beach St': 'Long Beach St.', 'MD Baltimore County': 'UMBC',
    'MTSU': 'Middle Tennessee', 'McNeese St': 'McNeese St.',
    'Miami OH': 'Miami (OH)', 'Michigan St': 'Michigan St.',
    'Mississippi St': 'Mississippi St.', 'Missouri St': 'Missouri St.',
    'Montana St': 'Montana St.', 'Morehead St': 'Morehead St.',
    'Morgan St': 'Morgan St.', "Mt St Mary's": "Mount St. Mary's",
    'Murray St': 'Murray St.', 'N Carolina St': 'N.C. State',
    'N Colorado': 'Northern Colorado', 'N Dakota St': 'North Dakota St.',
    'N Illinois': 'Northern Illinois', 'N Iowa': 'Northern Iowa',
    'N Kentucky': 'Northern Kentucky', 'NC A&T': 'North Carolina A&T',
    'NC Central': 'North Carolina Central', 'NW State': 'Northwestern St.',
    'New Mexico St': 'New Mexico St.', 'Nicholls St': 'Nicholls St.',
    'Norfolk St': 'Norfolk St.', 'Ohio St': 'Ohio St.',
    'Oklahoma St': 'Oklahoma St.', 'Oregon St': 'Oregon St.',
    'Penn St': 'Penn St.', 'Portland St': 'Portland St.',
    'Prairie View': 'Prairie View A&M', 'S Carolina St': 'South Carolina St.',
    'S Dakota St': 'South Dakota St.', 'S Illinois': 'Southern Illinois',
    'S Utah': 'Southern Utah', 'SE Louisiana': 'Southeastern Louisiana',
    'SE Missouri St': 'Southeast Missouri St.', 'SF Austin': 'Stephen F. Austin',
    'Sam Houston St': 'Sam Houston St.', 'San Diego St': 'San Diego St.',
    'Seattle': 'Seattle University', 'Southern Univ': 'Southern',
    'St Bonaventure': 'St. Bonaventure', 'St Francis NY': 'St. Francis (NY)',
    'St Francis PA': 'St. Francis (PA)', "St John's": "St. John's",
    "St Joseph's PA": "Saint Joseph's", "St Mary's CA": "Saint Mary's",
    "St Peter's": "Saint Peter's", 'TAM C. Christi': 'Texas A&M Corpus Chris',
    'TX Southern': 'Texas Southern', 'Tennessee St': 'Tennessee St.',
    'Tennessee Tech': 'Tennessee Tech', 'Texas St': 'Texas St.',
    'Tx A&M': 'Texas A&M', 'UC Davis': 'UC Davis', 'UC Irvine': 'UC Irvine',
    'UC Riverside': 'UC Riverside', 'UC Santa Barbara': 'UC Santa Barbara',
    'UNC Asheville': 'UNC Asheville', 'UNC Greensboro': 'UNC Greensboro',
    'UNC Wilmington': 'UNC Wilmington', 'UT Arlington': 'UT Arlington',
    'UT Martin': 'UT Martin', 'UT Rio Grande Valley': 'UTRGV',
    'Utah St': 'Utah St.', 'W Illinois': 'Western Illinois',
    'W Kentucky': 'Western Kentucky', 'W Michigan': 'Western Michigan',
    'Weber St': 'Weber St.', 'Wichita St': 'Wichita St.',
    'Wright St': 'Wright St.', 'Youngstown St': 'Youngstown St.',
    'MS Valley St': 'Mississippi Valley St.', 'SUNY Albany': 'Albany',
    'St Louis': 'Saint Louis', 'UT San Antonio': 'UTSA',
    'WKU': 'Western Kentucky', 'Washington St': 'Washington St.',
    'LIU Brooklyn': 'LIU', 'Miami (OH)': 'Miami OH',
    'Queens NC': 'Queens',
}

def map_name(name):
    return name_map.get(name, name)

print(f"Mappings defined: {len(name_map)}")

Mappings defined: 138


## 4. Build training dataset

In [5]:
team_lookup = teams.set_index('TeamID')['TeamName'].to_dict()
seed_lookup = seeds.set_index(['Season', 'TeamID'])['SeedNum'].to_dict()

results_filtered = results[results['Season'] >= 2010].copy()
results_filtered['WTeamName'] = results_filtered['WTeamID'].map(team_lookup).apply(map_name)
results_filtered['LTeamName'] = results_filtered['LTeamID'].map(team_lookup).apply(map_name)
results_filtered['WSeed'] = results_filtered.apply(lambda r: seed_lookup.get((r['Season'], r['WTeamID']), np.nan), axis=1)
results_filtered['LSeed'] = results_filtered.apply(lambda r: seed_lookup.get((r['Season'], r['LTeamID']), np.nan), axis=1)

# Merge Torvik base stats
results_filtered = results_filtered.merge(
    torvik_clean.rename(columns={col: f'W_{col}' for col in TORVIK_FEATURES}),
    left_on=['Season', 'WTeamName'], right_on=['year', 'team'], how='left'
).drop(columns=['year', 'team'])
results_filtered = results_filtered.merge(
    torvik_clean.rename(columns={col: f'L_{col}' for col in TORVIK_FEATURES}),
    left_on=['Season', 'LTeamName'], right_on=['year', 'team'], how='left'
).drop(columns=['year', 'team'])

print(f"Shape: {results_filtered.shape}")
print(f"Missing W_adjoe: {results_filtered['W_adjoe'].isna().sum()}")

Shape: (1001, 30)
Missing W_adjoe: 15


## 5. Load fffinal (shooting stats)

In [6]:
fffinal_dfs = []
for year in range(2010, 2027):
    url = f"http://barttorvik.com/{year}_fffinal.csv"
    response = requests.get(url, headers={"User-Agent": "Mozilla/5.0"}, timeout=15, verify=False)
    if response.status_code == 200:
        df = pd.read_csv(io.StringIO(response.text), quotechar='"', on_bad_lines='skip', index_col=False)
        df['year'] = year
        fffinal_dfs.append(df)
        print(f"Downloaded {year} — {len(df)} rows")
    else:
        print(f"Failed {year}")

fffinal = pd.concat(fffinal_dfs, ignore_index=True)
fffinal_clean = fffinal[['TeamName', 'year'] + FFFINAL_FEATURES].copy()
fffinal_clean['TeamName'] = fffinal_clean['TeamName'].apply(map_name)

results_filtered = results_filtered.merge(
    fffinal_clean.rename(columns={'TeamName': 'team', **{col: f'W_{col}' for col in FFFINAL_FEATURES}}),
    left_on=['Season', 'WTeamName'], right_on=['year', 'team'], how='left'
).drop(columns=['year', 'team'])
results_filtered = results_filtered.merge(
    fffinal_clean.rename(columns={'TeamName': 'team', **{col: f'L_{col}' for col in FFFINAL_FEATURES}}),
    left_on=['Season', 'LTeamName'], right_on=['year', 'team'], how='left'
).drop(columns=['year', 'team'])

print(f"Shape: {results_filtered.shape}")
print(f"Missing W_eFG%: {results_filtered['W_eFG%'].isna().sum()}")

Downloaded 2010 — 347 rows
Downloaded 2011 — 345 rows
Downloaded 2012 — 345 rows
Downloaded 2013 — 347 rows
Downloaded 2014 — 351 rows
Downloaded 2015 — 351 rows
Downloaded 2016 — 351 rows
Downloaded 2017 — 351 rows
Downloaded 2018 — 351 rows
Downloaded 2019 — 353 rows
Downloaded 2020 — 353 rows
Downloaded 2021 — 347 rows
Downloaded 2022 — 358 rows
Downloaded 2023 — 363 rows
Downloaded 2024 — 362 rows
Downloaded 2025 — 364 rows
Downloaded 2026 — 365 rows
Shape: (1001, 40)
Missing W_eFG%: 15


## 6. Load super_sked (variance/consistency stats)

In [7]:
col_names = [
    'muid','date','conmatch','matchup','prediction','ttq','conf','venue',
    'team1','t1oe','t1de','t1py','t1wp','t1propt',
    'team2','t2oe','t2de','t2py','t2wp','t2propt',
    'tpro','t1qual','t2qual','gp','result','tempo','possessions',
    't1pts','t2pts','winner','loser','t1adjt','t2adjt',
    't1adjo','t1adjd','t2adjo','t2adjd',
    'gamevalue','mismatch','blowout','t1elite','t2elite',
    'ord_date','t1ppp','t2ppp','gameppp',
    't1rk','t2rk','t1gs','t2gs','gamestats','overtimes',
    't1fun','t2fun','results','year'
]

sked_dfs = []
for year in range(2010, 2027):
    url = f"http://barttorvik.com/{year}_super_sked.csv"
    response = requests.get(url, headers={"User-Agent": "Mozilla/5.0"})
    if response.status_code == 200 and 'html' not in response.text[:100].lower():
        df = pd.read_csv(io.StringIO(response.text), quotechar='"', on_bad_lines='skip',
                         index_col=False, header=None, names=col_names[:56])
        df['year'] = year
        sked_dfs.append(df)
        print(f"Downloaded {year}")

sked = pd.concat(sked_dfs, ignore_index=True)
for col in ['t1oe','t1de','t2oe','t2de']:
    sked[col] = pd.to_numeric(sked[col], errors='coerce')

team1_games = sked[['year','team1','t1oe','t1de']].rename(columns={'team1':'team','t1oe':'game_oe','t1de':'game_de'})
team2_games = sked[['year','team2','t2oe','t2de']].rename(columns={'team2':'team','t2oe':'game_oe','t2de':'game_de'})
all_games = pd.concat([team1_games, team2_games]).dropna(subset=['game_oe','game_de'])

variance_stats = all_games.groupby(['year','team']).agg(
    oe_std=('game_oe','std'), de_std=('game_de','std'), games_played=('game_oe','count')
).reset_index()
variance_stats['team'] = variance_stats['team'].apply(map_name)

results_filtered = results_filtered.merge(
    variance_stats[['year','team','oe_std','de_std']].rename(columns={'oe_std':'W_oe_std','de_std':'W_de_std'}),
    left_on=['Season','WTeamName'], right_on=['year','team'], how='left'
).drop(columns=['year','team'])
results_filtered = results_filtered.merge(
    variance_stats[['year','team','oe_std','de_std']].rename(columns={'oe_std':'L_oe_std','de_std':'L_de_std'}),
    left_on=['Season','LTeamName'], right_on=['year','team'], how='left'
).drop(columns=['year','team'])

print(f"Shape: {results_filtered.shape}")
print(f"Missing W_oe_std: {results_filtered['W_oe_std'].isna().sum()}")

Downloaded 2010
Downloaded 2011
Downloaded 2012
Downloaded 2013
Downloaded 2014
Downloaded 2015
Downloaded 2016
Downloaded 2017
Downloaded 2018
Downloaded 2019
Downloaded 2020
Downloaded 2021
Downloaded 2022
Downloaded 2023
Downloaded 2024
Downloaded 2025
Downloaded 2026
Shape: (1001, 44)
Missing W_oe_std: 15


## 7. Load pre-tournament (Selection Sunday) ratings

In [8]:
selection_sundays = {
    2010:'20100314', 2011:'20110313', 2012:'20120311', 2013:'20130317',
    2014:'20140316', 2015:'20150315', 2016:'20160313', 2017:'20170312',
    2018:'20180311', 2019:'20190317', 2020:'20200315', 2021:'20210314',
    2022:'20220313', 2023:'20230312', 2024:'20240317', 2025:'20250316',
    2026:'20260315',
}

pretourney_dfs = []
for year in range(2010, 2027):
    date = selection_sundays[year]
    url = f"http://barttorvik.com/timemachine/team_results/{date}_team_results.json.gz"
    try:
        response = requests.get(url, headers={"User-Agent": "Mozilla/5.0"}, timeout=15)
        if response.status_code != 200:
            print(f"{year}: HTTP {response.status_code}, skipping")
            continue
        data = json.loads(response.content)
        rows = [{'team': e[1], 'year': year, 'pre_adjoe': e[4], 'pre_adjde': e[6],
                 'pre_barthag': e[8], 'pre_sos': e[10] if len(e) > 10 else None,
                 'pre_WAB': e[11] if len(e) > 11 else None} for e in data]
        pretourney_dfs.append(pd.DataFrame(rows))
        print(f"{year}: {len(rows)} teams loaded")
    except Exception as ex:
        print(f"{year}: ERROR — {ex}, skipping")

pretourney = pd.concat(pretourney_dfs, ignore_index=True)
pretourney['team'] = pretourney['team'].apply(map_name)

results_filtered = results_filtered.merge(
    pretourney.rename(columns={**{'team':'team','year':'year'}, **{c: f'W_{c}' for c in PRETOURNEY_FEATURES}}),
    left_on=['Season','WTeamName'], right_on=['year','team'], how='left'
).drop(columns=['year','team'])
results_filtered = results_filtered.merge(
    pretourney.rename(columns={**{'team':'team','year':'year'}, **{c: f'L_{c}' for c in PRETOURNEY_FEATURES}}),
    left_on=['Season','LTeamName'], right_on=['year','team'], how='left'
).drop(columns=['year','team'])

# Fallback: fill missing pre-tourney stats with season averages
for side, stat, fallback in [
    ('W','pre_adjoe','adjoe'), ('W','pre_adjde','adjde'), ('W','pre_barthag','barthag'),
    ('W','pre_sos','sos'), ('W','pre_WAB','WAB'),
    ('L','pre_adjoe','adjoe'), ('L','pre_adjde','adjde'), ('L','pre_barthag','barthag'),
    ('L','pre_sos','sos'), ('L','pre_WAB','WAB'),
]:
    results_filtered[f'{side}_{stat}'] = results_filtered[f'{side}_{stat}'].fillna(results_filtered[f'{side}_{fallback}'])

print(f"Shape: {results_filtered.shape}")
print(f"pre_ columns: {[c for c in results_filtered.columns if 'pre_' in c]}")
print(f"Missing pre stats: {results_filtered[['W_pre_adjoe','L_pre_adjoe']].isna().sum().to_dict()}")

2010: 347 teams loaded
2011: 345 teams loaded
2012: 345 teams loaded
2013: ERROR — ('Received response with content-encoding: gzip, but failed to decode it.', error('Error -3 while decompressing data: incorrect data check')), skipping
2014: 351 teams loaded
2015: 351 teams loaded
2016: 351 teams loaded
2017: 351 teams loaded
2018: 351 teams loaded
2019: 353 teams loaded
2020: 353 teams loaded
2021: 347 teams loaded
2022: 358 teams loaded
2023: 363 teams loaded
2024: 362 teams loaded
2025: 364 teams loaded
2026: 365 teams loaded
Shape: (1001, 54)
pre_ columns: ['W_pre_adjoe', 'W_pre_adjde', 'W_pre_barthag', 'W_pre_sos', 'W_pre_WAB', 'L_pre_adjoe', 'L_pre_adjde', 'L_pre_barthag', 'L_pre_sos', 'L_pre_WAB']
Missing pre stats: {'W_pre_adjoe': 15, 'L_pre_adjoe': 19}


## 8. Build training data & compute residual variance features

In [9]:
from xgboost import XGBClassifier
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LinearRegression

training_data = results_filtered.dropna().copy()
print(f"Training rows: {len(training_data)}")

# Compute difference features
for f in ALL_FEATURES:
    training_data[f'{f}_diff'] = training_data[f'W_{f}'] - training_data[f'L_{f}']
training_data['seed_diff'] = training_data['WSeed'] - training_data['LSeed']

# Randomly flip half the rows so model learns from both perspectives
np.random.seed(42)
flip = np.random.rand(len(training_data)) > 0.5

FEATURE_COLS = [f'{f}_diff' for f in ALL_FEATURES] + ['seed_diff']
for col in FEATURE_COLS:
    training_data.loc[flip, col] = -training_data.loc[flip, col]
training_data['label'] = (~flip).astype(int)
y = training_data['label']

# Compute residual variance (regress oe_std/de_std against barthag to remove multicollinearity)
for side in ['W', 'L']:
    for stat in ['oe_std', 'de_std']:
        X_reg = training_data[[f'{side}_barthag']].values
        y_reg = training_data[f'{side}_{stat}'].values
        reg = LinearRegression().fit(X_reg, y_reg)
        training_data[f'{side}_{stat}_resid'] = y_reg - reg.predict(X_reg)

training_data['oe_std_resid_diff'] = training_data['W_oe_std_resid'] - training_data['L_oe_std_resid']
training_data['de_std_resid_diff'] = training_data['W_de_std_resid'] - training_data['L_de_std_resid']
training_data.loc[flip, 'oe_std_resid_diff'] = -training_data.loc[flip, 'oe_std_resid_diff']
training_data.loc[flip, 'de_std_resid_diff'] = -training_data.loc[flip, 'de_std_resid_diff']

print("Residuals computed.")

Training rows: 967
Residuals computed.


## 9. Train residual model

In [10]:
X_resid = training_data[FEATURE_COLS_RESID]

model_resid = XGBClassifier(
    n_estimators=200, max_depth=3, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8, random_state=42, eval_metric='logloss'
)

scores_resid = cross_val_score(model_resid, X_resid, y, cv=5, scoring='roc_auc')
print(f"Cross-validation AUC: {scores_resid.mean():.3f} (+/- {scores_resid.std():.3f})")
print(f"Scores: {scores_resid.round(3)}")

model_resid.fit(X_resid, y)
print("Model trained!")

importance = pd.DataFrame({
    'feature': FEATURE_COLS_RESID,
    'importance': model_resid.feature_importances_
}).sort_values('importance', ascending=False)
print("\nTop 10 features:")
print(importance.head(10).to_string(index=False))

Cross-validation AUC: 0.882 (+/- 0.026)
Scores: [0.891 0.873 0.919 0.886 0.84 ]
Model trained!

Top 10 features:
          feature  importance
     barthag_diff    0.200981
       adjoe_diff    0.063435
         sos_diff    0.055865
     pre_sos_diff    0.053420
      Opp DE_diff    0.050528
       adjde_diff    0.047247
     pre_WAB_diff    0.045639
   pre_adjde_diff    0.043719
 pre_barthag_diff    0.043083
oe_std_resid_diff    0.036996


In [11]:
import joblib
model_resid = joblib.load('/Users/arnavgupta/Desktop/mm26/model_resid.joblib')
print("Model loaded!")

Model loaded!


## 10. Load 2026 matchups & generate predictions

In [13]:
matchups = pd.read_csv("/Users/arnavgupta/Desktop/mm26/2026_Potential_Matchups.csv")
matchups['HigherSeedName'] = matchups['HigherSeed'].apply(map_name)
matchups['LowerSeedName']  = matchups['LowerSeed'].apply(map_name)

# 2026 stat tables
torvik_2026    = torvik[torvik['year'] == 2026][['team'] + BASE_FEATURES].copy()
fffinal_2026   = fffinal[fffinal['year'] == 2026][['TeamName'] + FFFINAL_FEATURES].copy().rename(columns={'TeamName':'team'})
fffinal_2026['team']   = fffinal_2026['team'].apply(map_name)
variance_2026  = variance_stats[variance_stats['year'] == 2026][['team','oe_std','de_std']].copy()
variance_2026['team']  = variance_2026['team'].apply(map_name)
pretourney_2026= pretourney[pretourney['year'] == 2026][['team'] + PRETOURNEY_FEATURES].copy()

name_map['Miami OH'] = 'Miami OH'

matchups['HigherSeedName'] = matchups['HigherSeed'].apply(map_name)
matchups['LowerSeedName'] = matchups['LowerSeed'].apply(map_name)

# Fix Miami OH naming in 2026 tables
for df in [fffinal_2026, variance_2026, pretourney_2026]:
    df['team'] = df['team'].replace({'Miami (OH)': 'Miami OH'})

def merge_stats(matchups, stat_df, feats, seed_col, prefix):
    return matchups.merge(
        stat_df.rename(columns={f: f'{prefix}{f}' for f in feats}),
        left_on=seed_col, right_on='team', how='left'
    ).drop(columns=['team'], errors='ignore')

for df, feats in [(torvik_2026, BASE_FEATURES), (fffinal_2026, FFFINAL_FEATURES),
                  (variance_2026, VARIANCE_FEATURES), (pretourney_2026, PRETOURNEY_FEATURES)]:
    matchups = merge_stats(matchups, df, feats, 'HigherSeedName', 'H_')
for df, feats in [(torvik_2026, BASE_FEATURES), (fffinal_2026, FFFINAL_FEATURES),
                  (variance_2026, VARIANCE_FEATURES), (pretourney_2026, PRETOURNEY_FEATURES)]:
    matchups = merge_stats(matchups, df, feats, 'LowerSeedName', 'L_')

print(f"Shape: {matchups.shape}")
missing_h = matchups[[f'H_{f}' for f in ALL_FEATURES]].isna().sum()
print(f"Missing higher seed stats:\n{missing_h[missing_h > 0]}")

Shape: (2278, 51)
Missing higher seed stats:
Series([], dtype: int64)


In [14]:
name_map['NC State'] = 'N.C. State'
name_map['Miami (OH)'] = 'Miami OH'

matchups['HigherSeedName'] = matchups['HigherSeed'].apply(map_name)
matchups['LowerSeedName'] = matchups['LowerSeed'].apply(map_name)

In [15]:
# Compute residual variance for 2026 matchups
for side in ['H', 'L']:
    for stat in ['oe_std', 'de_std']:
        X_reg = training_data[['W_barthag']].values
        y_reg = training_data[f'W_{stat}'].values
        reg = LinearRegression().fit(X_reg, y_reg)
        matchups[f'{side}_{stat}_resid'] = (
            matchups[f'{side}_{stat}'].values - reg.predict(matchups[[f'H_barthag']].values)
        )

matchups['oe_std_resid_diff'] = matchups['H_oe_std_resid'] - matchups['L_oe_std_resid']
matchups['de_std_resid_diff'] = matchups['H_de_std_resid'] - matchups['L_de_std_resid']

for f in BASE_FEATURES + FFFINAL_FEATURES + PRETOURNEY_FEATURES:
    matchups[f'{f}_diff'] = matchups[f'H_{f}'] - matchups[f'L_{f}']
matchups['seed_diff'] = matchups['HigherSeedNum'] - matchups['LowerSeedNum']

X_pred = matchups[FEATURE_COLS_RESID]
matchups['Predictions'] = model_resid.predict_proba(X_pred)[:, 1]

print(f"Predictions generated for {len(matchups)} matchups")
print(f"\nPrediction distribution:")
print(matchups['Predictions'].describe().round(3))

Predictions generated for 2278 matchups

Prediction distribution:
count    2278.000
mean        0.700
std         0.185
min         0.085
25%         0.573
50%         0.735
75%         0.851
max         0.984
Name: Predictions, dtype: float64


## 11. Save submission

In [18]:
submission = matchups[['HigherSeed','HigherSeedID','HigherSeedNum',
                        'LowerSeed','LowerSeedID','LowerSeedNum','Predictions']].copy()
submission.to_csv("/Users/arnavgupta/Desktop/mm26/submission_residual.csv", index=False)
print("Submission saved to submission_residual.csv!")
print(f"Shape: {submission.shape}")
print(f"Missing predictions: {submission['Predictions'].isna().sum()}")

Submission saved to submission_residual.csv!
Shape: (2278, 7)
Missing predictions: 0


## 12. Bracket simulation

In [20]:
validated = pd.read_csv("/Users/arnavgupta/Desktop/mm26/submission.csv")

def get_prediction(team_a, team_b):
    row = validated[
        ((validated['HigherSeed'] == team_a) & (validated['LowerSeed'] == team_b)) |
        ((validated['HigherSeed'] == team_b) & (validated['LowerSeed'] == team_a))
    ]
    if len(row) == 0:
        return 0.5
    r = row.iloc[0]
    prob = r['Predictions']
    if r['HigherSeed'] == team_b:
        prob = 1 - prob
    return prob
def simulate_round(matchups_list):
    winners = []
    for team_a, team_b in matchups_list:
        prob = get_prediction(team_a, team_b)
        winner = team_a if prob >= 0.5 else team_b
        print(f"  {team_a} vs {team_b}: {prob:.1%} → {winner} wins")
        winners.append(winner)
    return winners

florida_opponent = 'Prairie View'
tenn_opponent = 'Miami OH'

round_of_64 = [
    # East
    ('Duke', 'Siena'), ('Ohio St', 'TCU'), ("St John's", 'Northern Iowa'), ('Kansas', 'Cal Baptist'),
    ('Louisville', 'South Florida'), ('Michigan St', 'N Dakota St'), ('UCLA', 'UCF'), ('Connecticut', 'Furman'),
    # South
    ('Florida', florida_opponent), ('Clemson', 'Iowa'), ('Vanderbilt', 'McNeese St'), ('Nebraska', 'Troy'),
    ('North Carolina', 'VCU'), ('Illinois', 'Penn'), ("Saint Mary's", 'Texas A&M'), ('Houston', 'Idaho'),
    # West
    ('Arizona', 'LIU'), ('Villanova', 'Utah St'), ('Wisconsin', 'High Point'), ('Arkansas', 'Hawaii'),
    ('BYU', 'Texas'), ('Gonzaga', 'Kennesaw St.'), ('Miami FL', 'Missouri'), ('Purdue', 'Queens'),
    # Midwest
    ('Michigan', 'Howard'), ('Georgia', 'Saint Louis'), ('Texas Tech', 'Akron'), ('Alabama', 'Hofstra'),
    ('Tennessee', tenn_opponent), ('Virginia', 'Wright St.'), ('Kentucky', 'Santa Clara'), ('Iowa St', 'Tennessee St.'),
]

print("\n" + "=" * 60 + "\nROUND OF 64\n" + "=" * 60)
r64 = simulate_round(round_of_64)
r32_matchups = [(r64[i], r64[i+1]) for i in range(0, len(r64), 2)]

print("\n" + "=" * 60 + "\nROUND OF 32\n" + "=" * 60)
r32 = simulate_round(r32_matchups)
s16_matchups = [(r32[i], r32[i+1]) for i in range(0, len(r32), 2)]

print("\n" + "=" * 60 + "\nSWEET 16\n" + "=" * 60)
s16 = simulate_round(s16_matchups)
e8_matchups = [(s16[i], s16[i+1]) for i in range(0, len(s16), 2)]

print("\n" + "=" * 60 + "\nELITE 8\n" + "=" * 60)
e8 = simulate_round(e8_matchups)
ff_matchups = [(e8[i], e8[i+1]) for i in range(0, len(e8), 2)]

print("\n" + "=" * 60 + "\nFINAL FOUR\n" + "=" * 60)
ff = simulate_round(ff_matchups)

print("\n" + "=" * 60 + "\nCHAMPIONSHIP\n" + "=" * 60)
champion = simulate_round([(ff[0], ff[1])])
print("\n" + "=" * 60)
print(f"CHAMPION: {champion[0]}")
print("=" * 60)


ROUND OF 64
  Duke vs Siena: 92.9% → Duke wins
  Ohio St vs TCU: 93.2% → Ohio St wins
  St John's vs Northern Iowa: 91.9% → St John's wins
  Kansas vs Cal Baptist: 93.7% → Kansas wins
  Louisville vs South Florida: 72.7% → Louisville wins
  Michigan St vs N Dakota St: 84.2% → Michigan St wins
  UCLA vs UCF: 79.0% → UCLA wins
  Connecticut vs Furman: 97.1% → Connecticut wins
  Florida vs Prairie View: 94.7% → Florida wins
  Clemson vs Iowa: 13.2% → Iowa wins
  Vanderbilt vs McNeese St: 96.1% → Vanderbilt wins
  Nebraska vs Troy: 95.9% → Nebraska wins
  North Carolina vs VCU: 79.8% → North Carolina wins
  Illinois vs Penn: 98.5% → Illinois wins
  Saint Mary's vs Texas A&M: 50.0% → Saint Mary's wins
  Houston vs Idaho: 97.3% → Houston wins
  Arizona vs LIU: 50.0% → Arizona wins
  Villanova vs Utah St: 51.5% → Villanova wins
  Wisconsin vs High Point: 92.3% → Wisconsin wins
  Arkansas vs Hawaii: 95.9% → Arkansas wins
  BYU vs Texas: 47.2% → Texas wins
  Gonzaga vs Kennesaw St.: 50.0% → Go

In [ ]:
validated = pd.read_csv("/Users/arnavgupta/Desktop/mm26/submission.csv")
print("Validated submission:")
print(validated['Predictions'].describe().round(3))

print("\nCurrent model:")
print(matchups['Predictions'].describe().round(3))

# Check a few specific matchups
test_pairs = [
    ('Duke', 'Michigan'),
    ('Florida', 'Connecticut'),
    ('Arizona', 'Purdue'),
    ('Michigan', 'Tennessee'),
]
print("\nMatchup comparison:")
for h, l in test_pairs:
    val_row = validated[((validated['HigherSeed']==h) & (validated['LowerSeed']==l)) |
                        ((validated['HigherSeed']==l) & (validated['LowerSeed']==h))]
    cur_row = matchups[((matchups['HigherSeed']==h) & (matchups['LowerSeed']==l)) |
                       ((matchups['HigherSeed']==l) & (matchups['LowerSeed']==h))]
    if len(val_row) and len(cur_row):
        print(f"{h} vs {l}: validated={val_row.iloc[0]['Predictions']:.3f}, current={cur_row.iloc[0]['Predictions']:.3f}")